# Manage View Layers & Web Maps

This notebook creates **six hosted view layers** and corresponding **web maps** from an existing field data collection layer.

| # | View | Definition Query | Editable | Sharing |
|---|------|-----------------|----------|---------|
| 1 | **Collector** | `review_status IS NULL OR review_status = 'Needs Revision'` | Create + Update | Public |
| 2 | **Pending Review** | `review_status = 'Pending Review'` | Update | Org |
| 3 | **In Review** | `review_status = 'In Review'` | Update | Org |
| 4 | **Approved (internal)** | `review_status = 'Approved'` | Update | Org |
| 5 | **Approved (public)** | `review_status = 'Approved'` | None (read-only) | Public |
| 6 | **Rejected** | `review_status = 'Rejected'` | Update | Org |

**Workflow**: Collector (NULL) → Arcade sets `Pending Review` on save → AI notebook enriches → `In Review` → Approve / Reject / Needs Revision → Needs Revision returns to Collector view

## Step 1 – Connect to ArcGIS Online

In [4]:
import getpass, urllib3
from arcgis.gis import GIS

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

portal_url = input("Portal URL [https://www.arcgis.com]: ").strip() or "https://www.arcgis.com"
username   = input("Username: ").strip()
password   = getpass.getpass("Password: ")

gis = GIS(portal_url, username, password, verify_cert=False)
print(f"Logged in as {gis.users.me.username} ({gis.url})")

Setting `verify_cert` to False is a security risk, use at your own risk.


Logged in as ralouta.aiddev (https://EsriAidDev.maps.arcgis.com)


## Step 2 – Load the Field Collection Layer

In [5]:
import json
from pathlib import Path
from arcgis.features import FeatureLayerCollection

# Auto-load base layer ID from registry if available
_registry_path = Path("..") / "item_registry.json"
item_id = None

if _registry_path.exists():
    _reg = json.loads(_registry_path.read_text())
    item_id = _reg.get("base_layer", {}).get("item_id")
    if item_id:
        print(f"Auto-loaded base layer ID from registry: {item_id}")

if not item_id:
    item_id = input("Field Collection Layer Item ID: ").strip()

flc_item = gis.content.get(item_id)
assert flc_item is not None, f"Item {item_id} not found."
print(f"Loaded: {flc_item.title} ({flc_item.type})")

flc = FeatureLayerCollection.fromitem(flc_item)
base_layer = flc.layers[0]
print(f"Layer: {base_layer.properties.name}  |  Features: {base_layer.query(return_count_only=True)}")

# Get all field names for reference
_all_fields = [f['name'] for f in base_layer.properties.fields]
print(f"Fields: {_all_fields}")

Auto-loaded base layer ID from registry: 59d38499beb34fc1b2db99571c899e05
Loaded: Plant Identification - AI (Feature Service)
Layer: Plant_Observations  |  Features: 2616
Fields: ['OBJECTID', 'GlobalID', 'common_name', 'latin_name', 'family', 'plant_type', 'condition', 'height_m', 'canopy_width_m', 'observation_date', 'observer', 'notes', 'review_status', 'reviewed_by', 'review_date', 'revision_notes', 'created_user', 'created_date', 'last_edited_user', 'last_edited_date']


## Step 3 – Resolve Folder

Views and web maps will be stored in the same folder as the source item.

In [6]:
# Use same folder as source item
source_folder = flc_item.ownerFolder
print(f"Target folder: {source_folder or '(Root)'}")

Target folder: 01b4f5dcec9948f6b21c39a6f3fc1e49


## Step 4 – Configure Field Collector Visibility

Select which fields the **Field Collector** view should expose, and which of those should be **read-only** (visible but not editable by collectors). System fields (`objectid`, `globalid`) are always included.

In [7]:
import ipywidgets as widgets
from IPython.display import display

# System fields always included (not shown in selection)
_SYSTEM_FIELDS = {"objectid", "globalid"}

# Fields to pre-select as visible/read-only by default
# Collector sees: review_status (read-only), revision_notes, notes (field notes)
# All AI-populated fields (common_name, latin_name, canopy_width_m, etc.) are hidden
_DEFAULT_VISIBLE = {"review_status", "revision_notes", "notes"}
_DEFAULT_READONLY = {"review_status"}

# Build field list from the base layer (excluding system fields from UI)
_base_fields = [f['name'] for f in base_layer.properties.fields]
_selectable_fields = [f for f in _base_fields if f.lower() not in _SYSTEM_FIELDS]

# Build a checkbox row per field: [FieldName] [✓ Visible] [✓ Read-only]
_field_checkboxes = {}  # field_name -> (visible_cb, readonly_cb)

_header = widgets.HBox([
    widgets.Label(value="Field", layout=widgets.Layout(width="200px", font_weight="bold")),
    widgets.Label(value="Visible", layout=widgets.Layout(width="100px", font_weight="bold")),
    widgets.Label(value="Read-only", layout=widgets.Layout(width="100px", font_weight="bold")),
])

_rows = [_header]
for fname in _selectable_fields:
    _vis_default = fname.lower() in _DEFAULT_VISIBLE
    _ro_default = fname.lower() in _DEFAULT_READONLY

    vis_cb = widgets.Checkbox(value=_vis_default, indent=False, layout=widgets.Layout(width="100px"))
    ro_cb = widgets.Checkbox(value=_ro_default, indent=False, layout=widgets.Layout(width="100px"))

    # Read-only only available when visible; auto-uncheck if visibility toggled off
    ro_cb.disabled = not _vis_default
    def _make_vis_handler(v_cb, r_cb):
        def _handler(change):
            if not change["new"]:
                r_cb.value = False
            r_cb.disabled = not change["new"]
        return _handler
    vis_cb.observe(_make_vis_handler(vis_cb, ro_cb), names="value")

    _field_checkboxes[fname] = (vis_cb, ro_cb)
    _rows.append(widgets.HBox([
        widgets.Label(value=fname, layout=widgets.Layout(width="200px")),
        vis_cb,
        ro_cb,
    ]))

print("Configure Field Collector view visibility:")
print("  System fields (objectid, globalid) are always included.\n")
display(widgets.VBox(_rows))

Configure Field Collector view visibility:
  System fields (objectid, globalid) are always included.



In [13]:
import re
import time
from arcgis.features import FeatureLayer

base_title = flc_item.title
# Strip any trailing " - Field Collection" for cleaner names
_clean = re.sub(r'\s*-\s*Field Collection$', '', base_title, flags=re.IGNORECASE)

# create_view `name` (service name) must be alphanumeric + underscores only
def _safe_name(label):
    return re.sub(r'[^A-Za-z0-9_]', '_', label).strip('_')

def _get_view_layer(view_item, gis, retries=20, delay=8):
    """Re-fetch a view item and return its first layer, retrying if not yet propagated."""
    for attempt in range(retries):
        item = gis.content.get(view_item.id)
        # Try FeatureLayerCollection first
        try:
            flc_v = FeatureLayerCollection.fromitem(item)
            if flc_v.layers:
                return item, flc_v.layers[0]
        except Exception:
            pass
        # Try direct FeatureLayer from URL
        if item.url:
            try:
                lyr = FeatureLayer(item.url + "/0", gis=gis)
                _ = lyr.properties
                return item, lyr
            except Exception:
                pass
        if attempt == 0:
            print(f"  View URL: {getattr(item, 'url', '(none)')}")
        print(f"  Waiting for view to propagate... ({attempt + 1}/{retries})")
        time.sleep(delay)
    raise RuntimeError(
        f"View {view_item.title} has no layers after {retries} retries ({retries * delay}s). "
        f"URL: {getattr(item, 'url', '(none)')}. "
        f"Check if the view item was created at: {gis.url}/home/item.html?id={view_item.id}"
    )

# ── Read field visibility from checkboxes ──────────────────────────────
_collector_visible = set()
_collector_readonly = set()
for fname, (vis_cb, ro_cb) in _field_checkboxes.items():
    if vis_cb.value:
        _collector_visible.add(fname.lower())
    if ro_cb.value:
        _collector_readonly.add(fname)

print(f"Collector visible fields: {sorted(_collector_visible)}")
print(f"Collector read-only fields: {sorted(_collector_readonly)}")

# Public view — fields safe for public consumption (no user/review data)
_PUBLIC_FIELDS = {
    "common_name", "latin_name", "family", "plant_type", "condition",
    "height_m", "canopy_width_m", "observation_date", "notes",
}

_view_layers = [base_layer]

# ── Helper: create a view, wait for propagation, optionally set definition query ──
def _create_view(title, updateable, capabilities, description, tags, snippet,
                 def_query=None, field_updates=None):
    """Create a view layer and apply optional definition query and field edits."""
    item = flc.manager.create_view(
        name=_safe_name(title),
        updateable=updateable,
        capabilities=capabilities,
        view_layers=_view_layers,
        description=description,
        tags=tags,
        snippet=snippet,
        folder=source_folder,
    )
    item.update(item_properties={"title": title})
    print(f"  Created view item: {item.id}  URL: {getattr(item, 'url', '(pending)')}")
    time.sleep(5)  # initial propagation delay
    item, lyr = _get_view_layer(item, gis)

    if def_query:
        lyr.manager.update_definition({"viewDefinitionQuery": def_query})

    if field_updates:
        lyr.manager.update_definition({"fields": field_updates})

    return item, lyr


# ── 1. Collector View ────────────────────────────────────────────────
_collector_title = f"{_clean} - Collector"
_collector_field_updates = []
for f in base_layer.properties.fields:
    fname = f["name"]
    if fname.lower() in _SYSTEM_FIELDS:
        continue
    if fname in _collector_readonly:
        _collector_field_updates.append({"name": fname, "editable": False})
    elif fname.lower() in _collector_visible:
        _collector_field_updates.append({"name": fname, "editable": True})

collector_view_item, collector_lyr = _create_view(
    _collector_title,
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing,Sync",
    description="Field Maps collection view. Shows features with NULL or Needs Revision status.",
    tags=f"{_clean}, Field Collection, Field Maps, Collector",
    snippet=f"Collector view for {_clean}",
    def_query="review_status IS NULL OR review_status = 'Needs Revision'",
    field_updates=_collector_field_updates,
)
print(f"✓ Collector view: {collector_view_item.title}  (ID: {collector_view_item.id})")
print(f"  → Filter: review_status IS NULL OR review_status = 'Needs Revision'")
print(f"  → Editable: {[f['name'] for f in _collector_field_updates if f.get('editable')]}")
print(f"  → Read-only: {[f['name'] for f in _collector_field_updates if not f.get('editable')]}")

# ── 2. Pending Review View ──────────────────────────────────────────
_pending_title = f"{_clean} - Pending Review"
pending_view_item, pending_lyr = _create_view(
    _pending_title,
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing",
    description="Features awaiting AI enrichment or manual review. Status = 'Pending Review'.",
    tags=f"{_clean}, Pending Review, AI",
    snippet=f"Pending Review view for {_clean}",
    def_query="review_status = 'Pending Review'",
)
print(f"✓ Pending Review view: {pending_view_item.title}  (ID: {pending_view_item.id})")

# ── 3. In Review View ───────────────────────────────────────────────
_in_review_title = f"{_clean} - In Review"
in_review_view_item, in_review_lyr = _create_view(
    _in_review_title,
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing",
    description="AI-enriched features ready for human review. Status = 'In Review'.",
    tags=f"{_clean}, In Review, Approver",
    snippet=f"In Review view for {_clean}",
    def_query="review_status = 'In Review'",
)
print(f"✓ In Review view: {in_review_view_item.title}  (ID: {in_review_view_item.id})")

# ── 4. Approved (internal) View ─────────────────────────────────────
_approved_int_title = f"{_clean} - Approved"
approved_int_view_item, approved_int_lyr = _create_view(
    _approved_int_title,
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing",
    description="Approved records — internal admin view with full edit access.",
    tags=f"{_clean}, Approved, Internal",
    snippet=f"Approved (internal) view for {_clean}",
    def_query="review_status = 'Approved'",
)
print(f"✓ Approved (internal) view: {approved_int_view_item.title}  (ID: {approved_int_view_item.id})")

# ── 5. Public View (approved, read-only) ────────────────────────────
_public_title = _clean  # clean name without suffix for public-facing layer
public_view_item, public_lyr = _create_view(
    _public_title,
    updateable=False,
    capabilities="Query,Extract",
    description="Read-only public view of approved records. User/review fields hidden.",
    tags=f"{_clean}, Public, Approved",
    snippet=f"Public approved data view for {_clean}",
    def_query="review_status = 'Approved'",
)
# Hide non-public fields by setting them invisible via field definition
_public_hide = []
for f in public_lyr.properties.fields:
    fname = f["name"]
    if fname.lower() in _SYSTEM_FIELDS:
        continue
    if fname.lower() not in _PUBLIC_FIELDS:
        _public_hide.append({"name": fname, "visible": False})
if _public_hide:
    public_lyr.manager.update_definition({"fields": _public_hide})
print(f"✓ Public view: {public_view_item.title}  (ID: {public_view_item.id})")
print(f"  → Hidden fields: {[f['name'] for f in _public_hide]}")

# ── 6. Rejected View ────────────────────────────────────────────────
_rejected_title = f"{_clean} - Rejected"
rejected_view_item, rejected_lyr = _create_view(
    _rejected_title,
    updateable=True,
    capabilities="Create,Query,Update,Delete,Editing",
    description="Rejected records for admin review/deletion.",
    tags=f"{_clean}, Rejected",
    snippet=f"Rejected view for {_clean}",
    def_query="review_status = 'Rejected'",
)
print(f"✓ Rejected view: {rejected_view_item.title}  (ID: {rejected_view_item.id})")

print("\n── All 6 views created ──")

Collector visible fields: ['created_date', 'created_user', 'notes', 'observation_date', 'observer', 'review_status']
Collector read-only fields: []
  Created view item: 0dabb6710dc74f5c8b35dc014fa0a1d3  URL: https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification___AI___Collector/FeatureServer
✓ Collector view: Plant Identification - AI - Collector  (ID: 0dabb6710dc74f5c8b35dc014fa0a1d3)
  → Filter: review_status IS NULL OR review_status = 'Needs Revision'
  → Editable: ['observation_date', 'observer', 'notes', 'review_status', 'created_user', 'created_date']
  → Read-only: []
  Created view item: e5f50cea8a074acbaec7e58836fc7473  URL: https://services.arcgis.com/LG9Yn2oFqZi5PnO5/arcgis/rest/services/Plant_Identification___AI___Pending_Review/FeatureServer
✓ Pending Review view: Plant Identification - AI - Pending Review  (ID: e5f50cea8a074acbaec7e58836fc7473)
  Created view item: 1cf60b937e8c4afab4673d8b00994abf  URL: https://services.arcgis.com/LG9Yn2oF

## Step 5 – Set Sharing

- **Collector** + **Public** views → Public
- All other views → Org (internal workflow)

In [15]:
# Internal views → Org
for _label, _item in [
    ("Pending Review", pending_view_item),
    ("In Review", in_review_view_item),
    ("Approved (internal)", approved_int_view_item),
    ("Rejected", rejected_view_item),
]:
    _item.sharing.sharing_level = "ORG"
    print(f"{_label} sharing: ORG")

# Collector + Public views → Everyone
for _label, _item in [
    ("Collector", collector_view_item),
    ("Public", public_view_item),
]:
    _item.sharing.sharing_level = "EVERYONE"
    print(f"{_label} sharing: EVERYONE")

Pending Review sharing: ORG
In Review sharing: ORG
Approved (internal) sharing: ORG
Rejected sharing: ORG
Collector sharing: EVERYONE
Public sharing: EVERYONE


## Step 6 – Create Web Maps

Each view layer gets a purpose-built web map with appropriate symbology.

In [16]:
import ipywidgets as widgets
from IPython.display import display

# ── Collection Schemas ─────────────────────────────────────────────
# Each schema defines the drawing tool and any auto-calculated fields.
# Add new schemas here as needed (e.g., "Buildings", "Wetlands").

COLLECTION_SCHEMAS = {
    "Trees": {
        "description": "Tree canopy mapping — tap center, drag radius",
        "drawing_tool": "esriFeatureEditToolCircle",
    },
    "General (Polygon)": {
        "description": "Free-form polygon collection",
        "drawing_tool": "esriFeatureEditToolPolygon",
    },
}

_schema_dropdown = widgets.Dropdown(
    options=list(COLLECTION_SCHEMAS.keys()),
    value="Trees",
    description="Schema:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="350px"),
)

_schema_desc = widgets.Label(
    value=COLLECTION_SCHEMAS["Trees"]["description"],
    layout=widgets.Layout(width="400px"),
)

def _on_schema_change(change):
    _schema_desc.value = COLLECTION_SCHEMAS[change["new"]]["description"]

_schema_dropdown.observe(_on_schema_change, names="value")

display(widgets.HBox([_schema_dropdown, _schema_desc]))
print("Select the collection schema, then run the remaining cells.")

Select the collection schema, then run the remaining cells.


In [19]:
# ── Symbology: Point marker symbols (esriSMS) by review_status ──────

_status_colors = {
    "Pending Review":  [255, 170, 0, 200],    # Orange
    "In Review":       [163, 73, 164, 200],    # Purple
    "Approved":        [56, 168, 0, 200],      # Green
    "Rejected":        [255, 0, 0, 200],       # Red
    "Needs Revision":  [0, 112, 255, 200],     # Blue
}

def _build_status_renderer():
    return {
        "type": "uniqueValue",
        "field1": "review_status",
        "defaultSymbol": {
            "type": "esriSMS",
            "style": "esriSMSCircle",
            "color": [130, 130, 130, 160],
            "size": 8,
            "outline": {"type": "esriSLS", "style": "esriSLSSolid", "color": [80, 80, 80, 255], "width": 0.75},
        },
        "defaultLabel": "New (NULL)",
        "uniqueValueInfos": [
            {"value": status, "label": status, "symbol": {
                "type": "esriSMS", "style": "esriSMSCircle",
                "color": color, "size": 8,
                "outline": {"type": "esriSLS", "style": "esriSLSSolid", "color": [0, 0, 0, 255], "width": 1},
            }} for status, color in _status_colors.items()
        ],
    }

# Single-color renderers for status-specific views
def _build_single_renderer(color):
    return {
        "type": "simple",
        "symbol": {
            "type": "esriSMS",
            "style": "esriSMSCircle",
            "color": color,
            "size": 8,
            "outline": {"type": "esriSLS", "style": "esriSLSSolid", "color": [255, 255, 255, 255], "width": 1},
        },
    }

collector_renderer = _build_status_renderer()
pending_renderer = _build_single_renderer([255, 170, 0, 140])
in_review_renderer = _build_status_renderer()  # shows both In Review + any transition states
approved_renderer = _build_single_renderer([56, 168, 0, 140])
public_renderer = _build_single_renderer([56, 168, 0, 140])
rejected_renderer = _build_single_renderer([255, 0, 0, 140])

# ── Apply renderers to all views ───────────────────────────────────
_view_renderers = [
    ("Collector", collector_view_item, collector_renderer),
    ("Pending Review", pending_view_item, pending_renderer),
    ("In Review", in_review_view_item, in_review_renderer),
    ("Approved (internal)", approved_int_view_item, approved_renderer),
    ("Public", public_view_item, public_renderer),
    ("Rejected", rejected_view_item, rejected_renderer),
]

for _label, _v_item, _rend in _view_renderers:
    _v_item, _lyr = _get_view_layer(_v_item, gis)
    _lyr.manager.update_definition({"drawingInfo": {"renderer": _rend}})
    # Update local variables
    if _label == "Collector":
        collector_view_item, collector_lyr = _v_item, _lyr
    elif _label == "Pending Review":
        pending_view_item, pending_lyr = _v_item, _lyr
    elif _label == "In Review":
        in_review_view_item, in_review_lyr = _v_item, _lyr
    elif _label == "Approved (internal)":
        approved_int_view_item, approved_int_lyr = _v_item, _lyr
    elif _label == "Public":
        public_view_item, public_lyr = _v_item, _lyr
    elif _label == "Rejected":
        rejected_view_item, rejected_lyr = _v_item, _lyr
    print(f"✓ {_label} renderer applied")

# ── Ensure attachments are enabled on SOURCE layer first ───────────
_src_has_attach = getattr(base_layer.properties, "hasAttachments", False)
if not _src_has_attach:
    base_layer.manager.update_definition({"hasAttachments": True})
    print("→ Enabled attachments on SOURCE layer")
else:
    print(f"→ Source layer attachments: {_src_has_attach}")

# Enable attachments on all editable views
for _label, _v_item in [
    ("Collector", collector_view_item),
    ("Pending Review", pending_view_item),
    ("In Review", in_review_view_item),
    ("Approved (internal)", approved_int_view_item),
    ("Rejected", rejected_view_item),
]:
    _v_item, _lyr = _get_view_layer(_v_item, gis)
    _has_att = getattr(_lyr.properties, "hasAttachments", False)
    if not _has_att:
        _lyr.manager.update_definition({"hasAttachments": True})
        print(f"→ Enabled attachments on {_label} view")

# ── Verify Collector editing capabilities ──────────────────────────
collector_flc = FeatureLayerCollection.fromitem(gis.content.get(collector_view_item.id))
_svc_caps = collector_flc.properties.get("capabilities", "")
_required = {"Create", "Query", "Update", "Delete", "Editing", "Sync"}
_current = {c.strip() for c in _svc_caps.split(",")}
if not _required.issubset(_current):
    _new_caps = ",".join(sorted(_required | _current))
    collector_flc.manager.update_definition({"capabilities": _new_caps})
    print(f"→ Updated Collector service capabilities to: {_new_caps}")
else:
    print(f"→ Collector service capabilities OK: {_svc_caps}")

✓ Collector renderer applied
✓ Pending Review renderer applied
✓ In Review renderer applied
✓ Approved (internal) renderer applied
✓ Public renderer applied
✓ Rejected renderer applied
→ Source layer attachments: True
→ Collector service capabilities OK: Create,Delete,Query,Update,Editing,Sync


In [20]:
import json

# ── Read the selected schema ───────────────────────────────────────
_active_schema = COLLECTION_SCHEMAS[_schema_dropdown.value]
_drawing_tool = _active_schema["drawing_tool"]
print(f"Schema: {_schema_dropdown.value}")
print(f"  Drawing tool: {_drawing_tool}")

# ── Create Web Maps via REST API (reliable for Field Maps) ─────────

def _get_data_extent(view_lyr):
    """Query the layer's actual data extent (full extent of all features)."""
    ext = dict(view_lyr.properties.extent)
    sr = ext.get("spatialReference", {"wkid": 4326})
    return {
        "xmin": ext["xmin"], "ymin": ext["ymin"],
        "xmax": ext["xmax"], "ymax": ext["ymax"],
        "spatialReference": sr if isinstance(sr, dict) else {"wkid": sr},
    }

def _build_popup(view_lyr, field_config=None):
    """Build Field Maps-compatible popup from layer fields."""
    field_infos = []
    for f in view_lyr.properties.fields:
        fname = f["name"]
        fname_lower = fname.lower()

        if fname_lower in ("objectid", "globalid"):
            field_infos.append({
                "fieldName": fname,
                "label": f.get("alias", fname),
                "visible": False,
                "isEditable": False,
            })
            continue

        if field_config and fname in field_config:
            cfg = field_config[fname]
            vis = cfg.get("visible", False)
            edit = cfg.get("editable", False)
        elif field_config and fname_lower in {k.lower() for k in field_config}:
            cfg = next(v for k, v in field_config.items() if k.lower() == fname_lower)
            vis = cfg.get("visible", False)
            edit = cfg.get("editable", False)
        else:
            vis = True
            edit = f.get("editable", True)

        field_infos.append({
            "fieldName": fname,
            "label": f.get("alias", fname),
            "visible": vis,
            "isEditable": edit,
        })

    return {
        "title": "{review_status} — {common_name}",
        "fieldInfos": field_infos,
        "popupElements": [
            {"type": "fields"},
            {"type": "attachments", "displayType": "auto"},
        ],
    }


def _build_templates(view_lyr, drawing_tool=None):
    """Build feature templates for the layer using the schema's drawing tool."""
    existing = getattr(view_lyr.properties, "templates", None)
    if existing:
        return [dict(t) for t in existing]

    geom_type = getattr(view_lyr.properties, "geometryType", "esriGeometryPolygon")
    _fallback = {
        "esriGeometryPoint": "esriFeatureEditToolPoint",
        "esriGeometryPolygon": "esriFeatureEditToolPolygon",
        "esriGeometryPolyline": "esriFeatureEditToolLine",
    }
    tool = drawing_tool or _fallback.get(geom_type, "esriFeatureEditToolPolygon")

    attrs = {}
    for f in view_lyr.properties.fields:
        if f["name"].lower() in ("objectid", "globalid"):
            continue
        attrs[f["name"]] = None

    return [{
        "name": "New Observation",
        "description": "",
        "drawingTool": tool,
        "prototype": {"attributes": attrs},
    }]


def _create_webmap(view_item, view_lyr, title, snippet, field_config=None,
                   editable=True, offline=False, folder=None, renderer=None,
                   drawing_tool=None):
    """Create a Web Map item with operational layer, popup, templates, and formInfo."""
    data_extent = _get_data_extent(view_lyr)
    popup_info = _build_popup(view_lyr, field_config=field_config)

    geom_type = getattr(view_lyr.properties, "geometryType", "esriGeometryPolygon")

    layer_def = {"geometryType": geom_type}
    if editable:
        caps = "Create,Delete,Query,Update,Editing"
        if offline:
            caps += ",Sync"
        layer_def["capabilities"] = caps
        layer_def["templates"] = _build_templates(view_lyr, drawing_tool=drawing_tool)

    if renderer:
        layer_def["drawingInfo"] = {"renderer": renderer}

    _has_att = getattr(view_lyr.properties, "hasAttachments", False)
    if _has_att:
        layer_def["hasAttachments"] = True

    op_layer = {
        "id": f"layer_{view_item.id[:8]}",
        "title": view_item.title,
        "url": view_lyr.url,
        "itemId": view_item.id,
        "layerType": "ArcGISFeatureLayer",
        "visibility": True,
        "opacity": 1,
        "disablePopup": False,
        "popupInfo": popup_info,
    }
    if layer_def:
        op_layer["layerDefinition"] = layer_def

    # Build formInfo for Field Maps editing form
    if editable:
        form_elements = []

        for f in view_lyr.properties.fields:
            fname = f["name"]
            if fname.lower() in ("objectid", "globalid"):
                continue
            vis = True
            if field_config:
                if fname in field_config:
                    vis = field_config[fname].get("visible", False)
                elif fname.lower() in {k.lower() for k in field_config}:
                    cfg = next(v for k, v in field_config.items() if k.lower() == fname.lower())
                    vis = cfg.get("visible", False)
            if not vis:
                continue

            _f_editable = True
            if field_config:
                if fname in field_config:
                    _f_editable = field_config[fname].get("editable", True)
                elif fname.lower() in {k.lower() for k in field_config}:
                    _fc = next(v for k, v in field_config.items() if k.lower() == fname.lower())
                    _f_editable = _fc.get("editable", True)

            elem = {
                "type": "field",
                "fieldName": fname,
                "label": f.get("alias", fname),
            }
            if not _f_editable:
                elem["editable"] = False
            form_elements.append(elem)

        op_layer["formInfo"] = {
            "title": view_item.title,
            "formElements": form_elements,
        }

    webmap_json = {
        "operationalLayers": [op_layer],
        "baseMap": {
            "baseMapLayers": [
                {
                    "id": "World_Imagery",
                    "title": "World Imagery",
                    "url": "https://services.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer",
                    "layerType": "ArcGISTiledMapServiceLayer",
                    "visibility": True,
                    "opacity": 1,
                }
            ],
            "title": "Imagery",
        },
        "initialState": {
            "viewpoint": {
                "targetGeometry": data_extent,
            }
        },
        "version": "2.28",
    }

    if editable or offline:
        webmap_json["applicationProperties"] = {
            "editing": {"locationTracking": {"enabled": False}},
            "offline": {
                "editableLayers": {"enabled": True},
                "readOnlyLayers": {"enabled": False},
                "basemapSwitching": {"enabled": False},
                "syncContribution": {"enabled": False},
            },
        }

    ext = data_extent
    item_props = {
        "title": title,
        "type": "Web Map",
        "snippet": snippet,
        "tags": f"{_clean}, Web Map",
        "text": json.dumps(webmap_json),
        "extent": f"{ext['xmin']},{ext['ymin']},{ext['xmax']},{ext['ymax']}",
    }
    if folder:
        target_folder = gis.content.folders.get(folder)
        result = target_folder.add(item_props)
    else:
        result = gis.content.folders.get().add(item_props)
    wm_item = result.result() if hasattr(result, 'result') else result

    wm_item.update(data=json.dumps(webmap_json))

    kw = set(wm_item.typeKeywords or [])
    kw.discard("FieldMapsDisabled")
    kw.discard("CollectorDisabled")
    if editable:
        kw.add("Collector")
    wm_item.update(item_properties={"typeKeywords": list(kw)})

    saved = wm_item.get_data()
    ol = saved.get("operationalLayers", [{}])[0]
    has_form = "formInfo" in ol
    has_caps = "layerDefinition" in ol and "capabilities" in ol.get("layerDefinition", {})
    has_app  = "applicationProperties" in saved
    print(f"  → Webmap data saved (formInfo={has_form}, caps={has_caps}, appProps={has_app})")

    return wm_item

# ── Build field config for collector from checkbox selections ──────
_collector_field_config = {}
for fname, (vis_cb, ro_cb) in _field_checkboxes.items():
    _collector_field_config[fname] = {
        "visible": vis_cb.value,
        "editable": vis_cb.value and not ro_cb.value,
    }
print("Collector popup field config:")
for fn, cfg in _collector_field_config.items():
    if cfg["visible"]:
        edit_str = "editable" if cfg["editable"] else "read-only"
        print(f"  {fn}: visible, {edit_str}")

# Re-fetch all view layers for URL access
collector_view_item, collector_lyr = _get_view_layer(collector_view_item, gis)
pending_view_item, pending_lyr = _get_view_layer(pending_view_item, gis)
in_review_view_item, in_review_lyr = _get_view_layer(in_review_view_item, gis)
approved_int_view_item, approved_int_lyr = _get_view_layer(approved_int_view_item, gis)
public_view_item, public_lyr = _get_view_layer(public_view_item, gis)
rejected_view_item, rejected_lyr = _get_view_layer(rejected_view_item, gis)

# 1. Collector Web Map — editable, offline-enabled
collector_wm = _create_webmap(
    collector_view_item, collector_lyr,
    f"{_clean} - Collector Map",
    "Field Maps collection map. Shows NULL + Needs Revision features.",
    field_config=_collector_field_config,
    editable=True, offline=True, folder=source_folder,
    renderer=collector_renderer, drawing_tool=_drawing_tool,
)
print(f"✓ Collector Map: {collector_wm.title}  (ID: {collector_wm.id})")

# 2. Pending Review Web Map — editable (AI notebook target)
pending_wm = _create_webmap(
    pending_view_item, pending_lyr,
    f"{_clean} - Pending Review Map",
    "Features awaiting AI enrichment. Pending Review status.",
    editable=True, folder=source_folder,
    renderer=pending_renderer, drawing_tool=_drawing_tool,
)
print(f"✓ Pending Review Map: {pending_wm.title}  (ID: {pending_wm.id})")

# 3. In Review Web Map — editable (reviewer approves/rejects here)
in_review_wm = _create_webmap(
    in_review_view_item, in_review_lyr,
    f"{_clean} - In Review Map",
    "Reviewer map — approve, reject, or send back AI-enriched records.",
    editable=True, folder=source_folder,
    renderer=in_review_renderer, drawing_tool=_drawing_tool,
)
print(f"✓ In Review Map: {in_review_wm.title}  (ID: {in_review_wm.id})")

# 4. Approved (internal) Web Map — editable (admin corrections)
approved_int_wm = _create_webmap(
    approved_int_view_item, approved_int_lyr,
    f"{_clean} - Approved Map",
    "Internal approved records map — admin corrections.",
    editable=True, folder=source_folder,
    renderer=approved_renderer, drawing_tool=_drawing_tool,
)
print(f"✓ Approved Map: {approved_int_wm.title}  (ID: {approved_int_wm.id})")

# 5. Public Web Map — read-only
public_wm = _create_webmap(
    public_view_item, public_lyr,
    f"{_clean} Map",
    "Public read-only map of approved records.",
    editable=False, folder=source_folder,
    renderer=public_renderer,
)
print(f"✓ Public Map: {public_wm.title}  (ID: {public_wm.id})")

# 6. Rejected Web Map — editable (admin can requeue or delete)
rejected_wm = _create_webmap(
    rejected_view_item, rejected_lyr,
    f"{_clean} - Rejected Map",
    "Rejected records — admin review and deletion.",
    editable=True, folder=source_folder,
    renderer=rejected_renderer, drawing_tool=_drawing_tool,
)
print(f"✓ Rejected Map: {rejected_wm.title}  (ID: {rejected_wm.id})")

Schema: Trees
  Drawing tool: esriFeatureEditToolCircle
Collector popup field config:
  observation_date: visible, editable
  observer: visible, editable
  notes: visible, editable
  review_status: visible, editable
  created_user: visible, editable
  created_date: visible, editable
  → Webmap data saved (formInfo=True, caps=True, appProps=True)
✓ Collector Map: Plant Identification - AI - Collector Map  (ID: 509ab8606c5f41488d758603141f2a8f)
  → Webmap data saved (formInfo=True, caps=True, appProps=True)
✓ Pending Review Map: Plant Identification - AI - Pending Review Map  (ID: 13d848a1c6464135aee56078a369b818)
  → Webmap data saved (formInfo=True, caps=True, appProps=True)
✓ In Review Map: Plant Identification - AI - In Review Map  (ID: aeb10c31a0cd4db291da1d06f712f613)
  → Webmap data saved (formInfo=True, caps=True, appProps=True)
✓ Approved Map: Plant Identification - AI - Approved Map  (ID: a2bb145b8ddb4c37abcfeb7023af421e)
  → Webmap data saved (formInfo=False, caps=False, appPr

In [21]:
# ── Share web maps to match their view layers ──────────────────────
for _label, _wm in [
    ("Pending Review", pending_wm),
    ("In Review", in_review_wm),
    ("Approved (internal)", approved_int_wm),
    ("Rejected", rejected_wm),
]:
    _wm.sharing.sharing_level = "ORG"
    print(f"{_label} Map sharing: ORG")

for _label, _wm in [
    ("Collector", collector_wm),
    ("Public", public_wm),
]:
    _wm.sharing.sharing_level = "EVERYONE"
    print(f"{_label} Map sharing: EVERYONE")

Pending Review Map sharing: ORG
In Review Map sharing: ORG
Approved (internal) Map sharing: ORG
Rejected Map sharing: ORG
Collector Map sharing: EVERYONE
Public Map sharing: EVERYONE


In [22]:
import requests as _rq
import tempfile, os

# ── Generate thumbnails via ArcGIS Online Print Service ────────────
_print_url = gis.properties.helperServices.printTask.url
_export_url = _print_url.rstrip("/") + "/execute"

def _build_text_overlay(label, extent):
    """Create a graphics layer with a large text label overlaid on the map."""
    cx = (extent["xmin"] + extent["xmax"]) / 2
    cy = extent["ymin"] + (extent["ymax"] - extent["ymin"]) * 0.10
    sr = extent.get("spatialReference", {"wkid": 4326})

    return {
        "id": "label_overlay",
        "opacity": 1,
        "minScale": 0,
        "maxScale": 0,
        "featureCollection": {
            "layers": [{
                "layerDefinition": {
                    "name": "Label",
                    "geometryType": "esriGeometryPoint",
                },
                "featureSet": {
                    "geometryType": "esriGeometryPoint",
                    "features": [{
                        "geometry": {
                            "x": cx,
                            "y": cy,
                            "spatialReference": sr,
                        },
                        "symbol": {
                            "type": "esriTS",
                            "color": [255, 255, 255, 255],
                            "backgroundColor": [0, 0, 0, 170],
                            "haloColor": [0, 0, 0, 255],
                            "haloSize": 2,
                            "text": label,
                            "horizontalAlignment": "center",
                            "verticalAlignment": "middle",
                            "font": {
                                "size": 28,
                                "weight": "bold",
                                "family": "Arial",
                            },
                        },
                    }],
                },
            }],
        },
    }

def _set_thumbnail(wm_item, label):
    """Export the webmap with a text overlay as thumbnail — MAP_ONLY, no frame."""
    wm_data = wm_item.get_data()

    extent = None
    try:
        extent = wm_data["initialState"]["viewpoint"]["targetGeometry"]
    except (KeyError, TypeError):
        pass

    if not extent:
        item_ext = wm_item.extent
        if item_ext:
            extent = {
                "xmin": item_ext[0][0], "ymin": item_ext[0][1],
                "xmax": item_ext[1][0], "ymax": item_ext[1][1],
                "spatialReference": {"wkid": 4326},
            }

    if not extent:
        print("no extent available")
        return False

    _token = gis._con.token
    op_layers = wm_data.get("operationalLayers", [])
    for lyr in op_layers:
        if _token:
            lyr["token"] = _token

    text_layer = _build_text_overlay(label, extent)
    op_layers.append(text_layer)

    print_json = {
        "mapOptions": {
            "extent": extent,
            "showAttribution": False,
        },
        "operationalLayers": op_layers,
        "baseMap": wm_data.get("baseMap", {}),
        "exportOptions": {
            "outputSize": [600, 400],
            "dpi": 96,
        },
    }

    payload = {
        "Web_Map_as_JSON": json.dumps(print_json),
        "Format": "PNG32",
        "Layout_Template": "MAP_ONLY",
        "f": "json",
    }

    resp = gis._con.post(_export_url, payload, timeout=120)
    result = resp.json() if hasattr(resp, 'json') else resp

    img_url = None
    if isinstance(result, dict):
        for r in result.get("results", []):
            val = r.get("value", {})
            if isinstance(val, dict) and "url" in val:
                img_url = val["url"]
                break
        if not img_url and "value" in result:
            val = result["value"]
            if isinstance(val, dict):
                img_url = val.get("url")
            elif isinstance(val, str) and val.startswith("http"):
                img_url = val

    if not img_url:
        print(f"\n    Response: {json.dumps(result, indent=2)[:500]}")
        return False

    img_data = _rq.get(img_url, timeout=30).content

    with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
        f.write(img_data)
        tmp_path = f.name

    wm_item.update(thumbnail=tmp_path)
    os.unlink(tmp_path)
    return True

for _label, _wm in [
    ("Collector", collector_wm),
    ("Pending Review", pending_wm),
    ("In Review", in_review_wm),
    ("Approved", approved_int_wm),
    ("Public", public_wm),
    ("Rejected", rejected_wm),
]:
    print(f"  {_label} thumbnail ... ", end="", flush=True)
    try:
        ok = _set_thumbnail(_wm, _label)
        print("✓" if ok else "failed (see response above)")
    except Exception as e:
        print(f"⚠️ {e}")

print("Done.")

  Collector thumbnail ... ✓
  Pending Review thumbnail ... ✓
  In Review thumbnail ... ✓
  Approved thumbnail ... ✓
  Public thumbnail ... ✓
  Rejected thumbnail ... ✓
Done.


## Step 7 – Summary

In [ ]:
print("=" * 70)
print("VIEW LAYERS & WEB MAPS CREATED")
print("=" * 70)

_all_views = [
    ("Collector",            collector_view_item, collector_wm,    "Public"),
    ("Pending Review",       pending_view_item,   pending_wm,      "Org"),
    ("In Review",            in_review_view_item, in_review_wm,    "Org"),
    ("Approved (internal)",  approved_int_view_item, approved_int_wm, "Org"),
    ("Public",               public_view_item,    public_wm,       "Public"),
    ("Rejected",             rejected_view_item,  rejected_wm,     "Org"),
]

for label, v_item, wm_item, sharing in _all_views:
    print(f"\n── {label} ──")
    print(f"  View Layer:  {v_item.title}")
    print(f"  View ID:     {v_item.id}")
    print(f"  Web Map:     {wm_item.title}")
    print(f"  Web Map ID:  {wm_item.id}")
    print(f"  Sharing:     {sharing}")

print(f"\n── Key Item IDs ──")
print(f"  Collector View:       {collector_view_item.id}")
print(f"  Pending Review View:  {pending_view_item.id}")
print(f"  In Review View:       {in_review_view_item.id}")

print("\n── Next Steps ──")
print("1. On the Collector view, add a Field Maps Arcade calculation on review_status → 'Pending Review'")
print("2. Add the Collector Map to Field Maps for photo collection")
print("3. Run the AI enrichment notebook targeting the Pending Review view")
print("4. Use the In Review Map to approve / reject / send back records")

VIEW LAYERS & WEB MAPS CREATED

── Collector ──
  View Layer:  Plant Identification - AI - Collector
  View ID:     593fbb367f854fe8a56c2fdfe1d763d1
  Web Map:     Plant Identification - AI - Collector Map
  Web Map ID:  fa9fb22669884ea4bf5070b424854519
  Sharing:     Org

── Pending Review ──
  View Layer:  Plant Identification - AI - Pending Review
  View ID:     5f37c72ad9d14f25b2b122b07b26c6b5
  Web Map:     Plant Identification - AI - Pending Review Map
  Web Map ID:  515dcdef3c2d4d2d904fb065260362d3
  Sharing:     Org

── In Review ──
  View Layer:  Plant Identification - AI - In Review
  View ID:     8cf26e9a25b44eeb8a11b8d7f9a5f5dc
  Web Map:     Plant Identification - AI - In Review Map
  Web Map ID:  82a37adec0c24df89643e40139b9e331
  Sharing:     Org

── Approved (internal) ──
  View Layer:  Plant Identification - AI - Approved
  View ID:     8e842d27bb9945deb414dbf1b9153df2
  Web Map:     Plant Identification - AI - Approved Map
  Web Map ID:  7484bb7a9a064a41b8e230883207c2

In [ ]:
import json
from pathlib import Path

# Save view & web map IDs to item registry
_registry_path = Path("..") / "item_registry.json"

if _registry_path.exists():
    registry = json.loads(_registry_path.read_text())
    print(f"Loaded existing registry from {_registry_path.resolve()}")
else:
    registry = {}
    print("No existing registry found — creating new one")

registry["views"] = {}
registry["web_maps"] = {}

for label, v_item, wm_item, sharing in _all_views:
    key = label.lower().replace(" ", "_").replace("(", "").replace(")", "")
    registry["views"][key] = {
        "item_id": v_item.id,
        "title": v_item.title,
    }
    registry["web_maps"][key] = {
        "item_id": wm_item.id,
        "title": wm_item.title,
    }

_registry_path.write_text(json.dumps(registry, indent=2))
print(f"\nSaved item registry to {_registry_path.resolve()}")
print(json.dumps(registry, indent=2))